1. Train a simple neural network to classify whether a review is positive or negative using a small dataset (you can use any dataset of your choice), then save the trained model to a .h5 file using model.save('sentiment_model.h5').


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 7s 0us/step


In [2]:
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=200)
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=200)

In [3]:
model = keras.Sequential([
    layers.Embedding(input_dim=10000, output_dim=16, input_length=200),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(x_test, y_test)
)

In [4]:
model.save('sentiment_model.h5')

print("Model saved successfully as sentiment_model.h5")

Model saved successfully as sentiment_model.h5


2. Write code to load the 'sentiment_model.h5' file you saved and use it to predict the sentiment of three new sample reviews.

In [10]:
import tensorflow as tf
from tensorflow import keras

model = keras.models.load_model('sentiment_model.h5')

In [11]:
word_index = keras.datasets.imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 2s 1us/step


In [12]:
import numpy as np

def encode_review(text, word_index, max_len=200):
    words = text.lower().split()
    encoded = []

    for word in words:
        idx = word_index.get(word, 2)
        encoded.append(idx)

    encoded = keras.preprocessing.sequence.pad_sequences(
        [encoded],
        maxlen=max_len
    )

    return encoded
   

In [14]:
reviews = [
    "this movie was amazing and fantastic",
    "worst movie ever very boring",
    "it was okay not great"
]

for review in reviews:
    encoded_review = encode_review(review, word_index)

    prediction = model.predict(encoded_review)[0][0]

    sentiment = "Positive " if prediction > 0.5 else "Negative "

    print(review)
    print("Score:", prediction)
    print("Sentiment:", sentiment)
    print("------")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step
this movie was amazing and fantastic
Score: 0.49217457
Sentiment: Negative 
------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
worst movie ever very boring
Score: 0.49214864
Sentiment: Negative 
------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
it was okay not great
Score: 0.49210995
Sentiment: Negative 
------


3. Implement model checkpointing during training so that the model's weights are saved every time the validation accuracy improves.<br><br><em><strong>Hint:</strong> Use the ModelCheckpoint callback in Keras with save_best_only=True.</em>


In [15]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

c:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [16]:
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath='best_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

In [17]:
history = model.fit(
    x_train,
    y_train,
    epochs=10,
    validation_data=(x_test, y_test),
    callbacks=[checkpoint],
    verbose=1
)

Epoch 1/10
1868/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8735 - loss: 0.4314
Epoch 1: val_accuracy improved from None to 0.95930, saving model to best_model.h5


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9246 - loss: 0.2606 - val_accuracy: 0.9593 - val_loss: 0.1340
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9641 - loss: 0.1225
Epoch 2: val_accuracy improved from 0.95930 to 0.96630, saving model to best_model.h5


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9663 - loss: 0.1156 - val_accuracy: 0.9663 - val_loss: 0.1055
Epoch 3/10
1869/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9759 - loss: 0.0805
Epoch 3: val_accuracy improved from 0.96630 to 0.97490, saving model to best_model.h5


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9759 - loss: 0.0797 - val_accuracy: 0.9749 - val_loss: 0.0802
Epoch 4/10
1867/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9838 - loss: 0.0554
Epoch 4: val_accuracy improved from 0.97490 to 0.97730, saving model to best_model.h5


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9827 - loss: 0.0575 - val_accuracy: 0.9773 - val_loss: 0.0743
Epoch 5/10
1866/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9865 - loss: 0.0432
Epoch 5: val_accuracy improved from 0.97730 to 0.97740, saving model to best_model.h5


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9860 - loss: 0.0453 - val_accuracy: 0.9774 - val_loss: 0.0731
Epoch 6/10
1861/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9896 - loss: 0.0344
Epoch 6: val_accuracy improved from 0.97740 to 0.97850, saving model to best_model.h5


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9887 - loss: 0.0357 - val_accuracy: 0.9785 - val_loss: 0.0640
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9924 - loss: 0.0259
Epoch 7: val_accuracy did not improve from 0.97850
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9909 - loss: 0.0283 - val_accuracy: 0.9771 - val_loss: 0.0758
Epoch 8/10
1859/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9945 - loss: 0.0199
Epoch 8: val_accuracy improved from 0.97850 to 0.97910, saving model to best_model.h5


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9931 - loss: 0.0223 - val_accuracy: 0.9791 - val_loss: 0.0746
Epoch 9/10
1869/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9948 - loss: 0.0171
Epoch 9: val_accuracy did not improve from 0.97910
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9937 - loss: 0.0197 - val_accuracy: 0.9763 - val_loss: 0.0839
Epoch 10/10
1865/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9964 - loss: 0.0137
Epoch 10: val_accuracy did not improve from 0.97910
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.9953 - loss: 0.0156 - val_accuracy: 0.9781 - val_loss: 0.0848


5. Use ChatGPT or Copilot to help you write the code for saving both the model architecture and weights separately, then test loading them back and confirm the loaded model gives the same predictions as the original.

In [20]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(x_train, y_train, epochs=2, verbose=0)

In [21]:
model_json = model.to_json()

with open("model_architecture.json", "w") as json_file:
    json_file.write(model_json)